[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S24_no_supervisado.ipynb)

# Sesión 24 · Aprendizaje no supervisado

**Módulo 6: Negocio y extras** · ⏱️ Duración estimada: 60 a 75 minutos

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Agrupar clientes con K-means y explicar qué optimiza.
2. Explicar por qué hay que escalar antes de agrupar.
3. Elegir el número de grupos con el método del codo y la silueta.
4. Resumir muchas variables en dos con PCA para visualizar los grupos.
5. Interpretar y nombrar segmentos, y asignar clientes nuevos.

## 📋 Qué debes saber antes
Sesiones 17 y 21: la API de scikit-learn (`fit`, `predict`, `transform`) y `StandardScaler`.

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`, con los nombres de variables que se piden.
- Después ejecuta la celda ✅ **Verificar**. Si aparece ❌, lee el motivo, corrige y vuelve a verificar.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Genera los datos, aplica el estilo de gráficos y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión, aplica el estilo de gráficos y carga los verificadores.
import copy
import hashlib
import math
import statistics

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ---------- Estilo de los gráficos ----------
# Paleta categórica en orden fijo (validada para daltonismo) y tintas para textos y ejes.
AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948")
GRIS = "#c3c2b7"          # para lo que no es protagonista
TINTA = "#0b0b0b"         # textos principales
TINTA_2 = "#52514e"       # textos secundarios
FONDO = "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "axes.edgecolor": GRIS, "axes.labelcolor": TINTA_2, "text.color": TINTA,
    "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO]),
    "axes.titlesize": 13, "axes.titlelocation": "left", "axes.titleweight": "bold",
    "lines.linewidth": 2, "font.size": 11, "figure.dpi": 100,
    "axes.formatter.useoffset": False, "axes.formatter.limits": (-9, 9),   # sin notación científica (1e6)
})


# ---------- Datos de práctica: cómo usan el banco 2000 clientes ----------
_n = 2000
_grupo = rng.choice(4, _n, p=[0.3, 0.2, 0.3, 0.2])
_param = {
    "edad": [(28, 4), (62, 6), (42, 6), (38, 7)],
    "saldo_promedio": [(1500, 600), (30000, 9000), (8000, 3000), (14000, 5000)],
    "transacciones_mes": [(45, 10), (8, 3), (25, 6), (70, 15)],
    "uso_app": [(0.9, 0.06), (0.2, 0.1), (0.6, 0.12), (0.65, 0.12)],
    "gasto_tarjeta": [(1200, 350), (600, 250), (3000, 700), (2000, 600)],
    "pct_efectivo": [(0.1, 0.05), (0.6, 0.12), (0.3, 0.08), (0.5, 0.1)],
}
_d = {k: rng.normal(np.array([m for m, s in v])[_grupo], np.array([s for m, s in v])[_grupo]) for k, v in _param.items()}
clientes = pd.DataFrame({
    "edad": np.clip(_d["edad"], 18, 90).round().astype(int),
    "saldo_promedio": np.clip(_d["saldo_promedio"], 50, None).round(2),
    "transacciones_mes": np.clip(_d["transacciones_mes"], 1, None).round().astype(int),
    "uso_app": np.clip(_d["uso_app"], 0, 1).round(2),
    "gasto_tarjeta": np.clip(_d["gasto_tarjeta"], 0, None).round(2),
    "pct_efectivo": np.clip(_d["pct_efectivo"], 0, 1).round(2),
})
del _d, _grupo
clientes_nuevos = pd.DataFrame({
    "edad": [25, 66, 45, 30], "saldo_promedio": [900.0, 28000.0, 9500.0, 250000.0],
    "transacciones_mes": [50, 6, 22, 120], "uso_app": [0.95, 0.1, 0.55, 0.3],
    "gasto_tarjeta": [1000.0, 400.0, 3200.0, 9000.0], "pct_efectivo": [0.05, 0.7, 0.25, 0.9],
})
VARIABLES = list(clientes.columns)
K_VALORES = list(range(2, 9))

_D = copy.deepcopy({"clientes": clientes, "clientes_nuevos": clientes_nuevos})

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _norm(x):
    if isinstance(x, np.generic):
        x = x.item()
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(x, pd.Timestamp):
        return str(x)
    return x


def _mismo(a, b, tol=1e-6):
    a, b = _norm(a), _norm(b)
    if a is None or b is None:
        return a is None and b is None
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return isinstance(a, (int, float)) and not isinstance(a, bool) and abs(a - b) <= tol
    return str(a) == str(b)


def _ser(r, nombre, valores, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.Series):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba una Series de pandas.")
        return
    if len(v) != len(valores):
        r.mal(f"`{nombre}` tiene {len(v)} elementos y se esperaban {len(valores)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    if all(_mismo(a, b, tol) for a, b in zip(v.tolist(), valores)):
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene el largo correcto pero sus valores no coinciden; {pista}.")


def _df(r, nombre, columnas, filas, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.DataFrame):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un DataFrame de pandas.")
        return
    cols = [str(c) for c in v.columns]
    if cols != columnas:
        faltan = [c for c in columnas if c not in cols]
        sobran = [c for c in cols if c not in columnas]
        if faltan or sobran:
            r.mal(f"A `{nombre}` le faltan las columnas {faltan} y le sobran {sobran}." if faltan and sobran else
                  (f"A `{nombre}` le faltan las columnas {faltan}." if faltan else f"En `{nombre}` sobran las columnas {sobran}."))
        else:
            r.mal(f"`{nombre}` tiene las columnas correctas pero en otro orden.")
        return
    if len(v) != len(filas):
        r.mal(f"`{nombre}` tiene {len(v)} filas y se esperaban {len(filas)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas de fila (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    bien = all(_mismo(a, b, tol) for fila_v, fila_e in zip(v.itertuples(index=False), filas) for a, b in zip(fila_v, fila_e))
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")


def _sin_cambios_df(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        if not (isinstance(actual, (pd.DataFrame, pd.Series)) and actual.equals(_D[n])):
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a cargarlos o ejecuta de nuevo el setup.")

def _hex(c):
    return mcolors.to_hex(c).lower()


def _texto(a, b):
    return isinstance(a, str) and a.strip().lower() == b.strip().lower()


def _grafico(r, nombre):
    ax = r.var(nombre)
    if ax is _FALTA:
        return None
    if not isinstance(ax, mpl.axes.Axes):
        r.mal(f"`{nombre}` es de tipo {type(ax).__name__} y se esperaba un eje de Matplotlib (lo que devuelve `plt.subplots()`).")
        return None
    return ax


def _rotulos(r, nombre, ax, titulo=None, xlabel=None, ylabel=None):
    titulo_actual = next((t for t in (ax.get_title(loc=l) for l in ("left", "center", "right")) if t.strip()), "")
    for que, genero, obtenido, esperado in (("el título", "correcto", titulo_actual, titulo),
                                            ("la etiqueta del eje x", "correcta", ax.get_xlabel(), xlabel),
                                            ("la etiqueta del eje y", "correcta", ax.get_ylabel(), ylabel)):
        if esperado is None:
            continue
        if _texto(obtenido, esperado):
            r.ok(f"En `{nombre}`, {que} es {genero}.")
        elif not obtenido.strip():
            r.mal(f"A `{nombre}` le falta {que}.")
        else:
            r.mal(f"En `{nombre}`, {que} dice {obtenido!r}; revisa el texto pedido.")


def _barras(ax):
    """Rectángulos de barras (sin el fondo del eje), en orden de dibujo."""
    return [p for p in ax.patches if isinstance(p, mpl.patches.Rectangle)]


def _cerca_lista(a, b, tol=1e-6):
    return len(a) == len(b) and all(abs(float(x) - float(y)) <= tol for x, y in zip(a, b))


def _formato(ax, eje, valor):
    fmt = (ax.yaxis if eje == "y" else ax.xaxis).get_major_formatter()
    return fmt(valor, 0)


def _esc_ref(A, B=None):
    """Escala B con la media y la desviación (ddof=0) de A, columna por columna, sin scikit-learn."""
    A = np.asarray(A, dtype=float)
    B = A if B is None else np.asarray(B, dtype=float)
    medias = [statistics.fmean(A[:, j]) for j in range(A.shape[1])]
    desv = [statistics.pstdev(A[:, j]) for j in range(A.shape[1])]
    return np.column_stack([(B[:, j] - medias[j]) / desv[j] for j in range(B.shape[1])])


def _dist2(X, C):
    """Distancias al cuadrado entre filas de X y de C, con la identidad |x - c|² = |x|² + |c|² - 2 x·c."""
    return np.maximum((X ** 2).sum(axis=1)[:, None] + (C ** 2).sum(axis=1)[None, :] - 2 * X @ C.T, 0)


def _km(X, k):
    from sklearn.cluster import KMeans
    return KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)


def _silueta(X, etiq):
    """Silueta media calculada con la matriz de distancias completa."""
    D = np.sqrt(np.maximum(_dist2(X, X), 0))
    etiq = np.asarray(etiq)
    grupos = sorted(set(etiq.tolist()))
    medias = np.column_stack([D[:, etiq == g].sum(axis=1) for g in grupos])
    tam = np.array([(etiq == g).sum() for g in grupos])
    s = np.zeros(len(X))
    for i in range(len(X)):
        propio = grupos.index(etiq[i])
        if tam[propio] == 1:
            continue
        a = medias[i, propio] / (tam[propio] - 1)
        b = min(medias[i, j] / tam[j] for j in range(len(grupos)) if j != propio)
        s[i] = (b - a) / max(a, b)
    return s


def _ari(a, b):
    """Índice de Rand ajustado con la tabla de contingencia y combinaciones."""
    a, b = list(a), list(b)
    n = len(a)
    tabla = {}
    for x, y in zip(a, b):
        tabla[(x, y)] = tabla.get((x, y), 0) + 1
    filas, cols = {}, {}
    for (x, y), c in tabla.items():
        filas[x] = filas.get(x, 0) + c
        cols[y] = cols.get(y, 0) + c
    s_ij = sum(math.comb(c, 2) for c in tabla.values())
    s_a = sum(math.comb(c, 2) for c in filas.values())
    s_b = sum(math.comb(c, 2) for c in cols.values())
    esperado = s_a * s_b / math.comb(n, 2)
    return (s_ij - esperado) / ((s_a + s_b) / 2 - esperado)


def _modelo_km(r, nombre, k):
    m = r.var(nombre)
    if m is _FALTA:
        return None
    if type(m).__name__ != "KMeans" or not hasattr(m, "cluster_centers_"):
        r.mal(f"`{nombre}` debería ser un `KMeans` ya entrenado.")
        return None
    if (m.n_clusters, m.random_state) != (k, 42) or m.n_init != 10:
        r.mal(f"`{nombre}` debería tener `n_clusters={k}`, `n_init=10` y `random_state=42`.")
        return None
    return m


def _X():
    return _esc_ref(clientes[VARIABLES].to_numpy(float))


def check_ejercicio_1():
    r = _Revision("Ejercicio 1 · Parte A")
    _sin_cambios_df(r, "clientes", "clientes_nuevos")
    X = _X()
    e = r.var("escalador")
    if e is not _FALTA:
        if type(e).__name__ != "StandardScaler" or not hasattr(e, "mean_") or len(e.mean_) != len(VARIABLES):
            r.mal("`escalador` debería ser un `StandardScaler` ajustado con `clientes[VARIABLES]`.")
        elif not _cerca_lista(e.mean_, [statistics.fmean(clientes[c]) for c in VARIABLES], 1e-6):
            r.mal("`escalador` no tiene las medias de `clientes[VARIABLES]`.")
        else:
            r.ok("`escalador` es correcto.")
    v = r.var("X_esc")
    if v is not _FALTA:
        v = np.asarray(v, dtype=float)
        if v.shape != X.shape:
            r.mal(f"`X_esc` tiene forma {v.shape} y se esperaba {X.shape}.")
        elif not np.allclose(v, X, atol=1e-6):
            r.mal("Los valores de `X_esc` no coinciden: transforma `clientes[VARIABLES]` con `escalador`.")
        else:
            r.ok("`X_esc` es correcto.")
    m = _modelo_km(r, "km4", 4)
    if m is not None:
        C = m.cluster_centers_
        if C.shape != (4, X.shape[1]):
            r.mal("`km4` debería entrenarse con `X_esc` (las 6 variables escaladas).")
        else:
            d = _dist2(X, C)
            cerca = d.argmin(axis=1)
            if (cerca != m.labels_).mean() > 0.001:
                r.mal("`km4` no parece entrenado con `X_esc`: sus grupos no coinciden con los centros más cercanos.")
            else:
                r.ok("`km4` agrupa `X_esc` en 4 grupos.")
                et = r.var("etiquetas")
                if et is not _FALTA:
                    et = np.asarray(et)
                    if et.shape != (len(X),) or not np.array_equal(et, cerca):
                        r.mal("`etiquetas` debería ser el grupo de cada cliente (`km4.labels_`).")
                    else:
                        r.ok("`etiquetas` es correcto.")
                cuenta = [int((cerca == g).sum()) for g in range(4)]
                _ser(r, "tamanos", cuenta, "cuántos clientes hay en cada grupo, con el número de grupo como índice ordenado", indice=[0, 1, 2, 3], tol=0)
                _esc(r, "inercia_4", float(d.min(axis=1).sum()), "la inercia de `km4` (`inertia_`)", tol=1e-3)
    r.fin()
    r = _Revision("Ejercicio 1 · Parte B")
    r.predicciones({
        "pred_numeros_cambian": "6c5fb3b25e6ba7dcf12155440e0c51b36a0492e24123968a10f8c31c304ebb6e",
    })
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2 · Parte A")
    m = _modelo_km(r, "km_crudo", 4)
    X_crudo = clientes[VARIABLES].to_numpy(float)
    if m is not None:
        cerca = _dist2(X_crudo, m.cluster_centers_).argmin(axis=1) if m.cluster_centers_.shape[1] == X_crudo.shape[1] else None
        if cerca is None or (cerca != m.labels_).mean() > 0.001 or np.abs(m.cluster_centers_).max() < 100:
            r.mal("`km_crudo` debería entrenarse con `clientes[VARIABLES]` **sin** escalar.")
        else:
            r.ok("`km_crudo` agrupa los datos sin escalar.")
            et = r.var("etiquetas_crudo")
            if et is not _FALTA:
                if not np.array_equal(np.asarray(et), cerca):
                    r.mal("`etiquetas_crudo` debería ser `km_crudo.labels_`.")
                else:
                    r.ok("`etiquetas_crudo` es correcto.")
            ref = globals().get("km4")
            if hasattr(ref, "cluster_centers_") and ref.cluster_centers_.shape == (4, X_crudo.shape[1]):
                esc = _dist2(_X(), ref.cluster_centers_).argmin(axis=1)
                _esc(r, "ari", _ari(esc, cerca), "`adjusted_rand_score(etiquetas, etiquetas_crudo)`", tol=1e-9)
            else:
                r.mal("Para revisar `ari` necesito `km4` del ejercicio 1.")
    desv = {c: statistics.stdev(clientes[c]) for c in VARIABLES}
    orden = sorted(desv, key=lambda c: -desv[c])
    _ser(r, "desv_por_variable", [desv[c] for c in orden], "la desviación estándar de cada variable sin escalar, de mayor a menor", indice=orden, tol=1e-6)
    r.fin()
    r = _Revision("Ejercicio 2 · Parte B")
    r.predicciones({
        "pred_domina": "956aaeedf135fefda02db2a12a836189212f35701b5c9cdc1a4d67e4e91da265",
    })
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3 · Parte A")
    X = _X()
    filas = []
    for k in K_VALORES:
        m = _km(X, k)
        filas.append([float(_dist2(X, m.cluster_centers_).min(axis=1).sum()), float(_silueta(X, m.labels_).mean())])
    _df(r, "evaluacion_k", ["inercia", "silueta"], filas, "un K-means por cada k de `K_VALORES` (con `n_init=10` y `random_state=42`) sobre `X_esc`", indice=K_VALORES, tol=1e-3)
    sil = [f[1] for f in filas]
    _esc(r, "k_silueta", K_VALORES[sil.index(max(sil))], "el k con mayor silueta", tol=0)
    for nombre, col, eje_y in (("ax_codo", 0, "Inercia"), ("ax_silueta", 1, "Silueta")):
        ax = _grafico(r, nombre)
        if ax is None:
            continue
        lineas = [l for l in ax.get_lines() if len(l.get_ydata()) == len(K_VALORES)]
        if not any(_cerca_lista(l.get_xdata(), K_VALORES) and _cerca_lista(l.get_ydata(), [f[col] for f in filas], 1e-3) for l in lineas):
            r.mal(f"`{nombre}` debería tener una línea con la columna `{['inercia', 'silueta'][col]}` de `evaluacion_k` según k.")
        else:
            r.ok(f"`{nombre}` es correcto.")
        _rotulos(r, nombre, ax, None, "k", eje_y)
    r.fin()
    r = _Revision("Ejercicio 3 · Parte B")
    r.predicciones({
        "pred_inercia_maxima_k": "399afcbd9cfc35e6944d699c5f5b3cbbcd510a3a44f8f08e8172055ba1a27cba",
    })
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4 · Parte A")
    m = globals().get("km4")
    if not hasattr(m, "cluster_centers_") or m.cluster_centers_.shape != (4, len(VARIABLES)):
        r.mal("Primero resuelve el ejercicio 1 (`km4`).")
        r.fin()
        return
    X = _X()
    et = _dist2(X, m.cluster_centers_).argmin(axis=1)
    cs = r.var("clientes_seg")
    if cs is not _FALTA:
        if not isinstance(cs, pd.DataFrame) or "cluster" not in cs or len(cs) != len(clientes) or not np.array_equal(cs["cluster"].to_numpy(), et):
            r.mal("`clientes_seg` debería ser una copia de `clientes` con la columna `cluster` (las etiquetas de `km4`).")
        elif not cs[VARIABLES].equals(clientes[VARIABLES]):
            r.mal("Las variables de `clientes_seg` deberían ser las originales, sin escalar.")
        else:
            r.ok("`clientes_seg` tiene el grupo de cada cliente.")
    filas, filas_z = [], []
    for g in range(4):
        idx = np.where(et == g)[0]
        filas.append([statistics.fmean(clientes[c].iloc[idx]) for c in VARIABLES] + [len(idx)])
        filas_z.append([statistics.fmean(X[idx, j]) for j in range(len(VARIABLES))])
    _df(r, "perfil", VARIABLES + ["clientes"], filas, "el promedio de cada variable original por grupo y la cantidad de clientes", indice=[0, 1, 2, 3], tol=1e-6)
    _df(r, "perfil_z", VARIABLES, filas_z, "el promedio de cada variable **escalada** por grupo", indice=[0, 1, 2, 3], tol=1e-6)
    nombres = r.var("nombres", dict)
    if nombres is not _FALTA:
        if sorted(nombres) != [0, 1, 2, 3]:
            r.mal("`nombres` debería tener una clave por grupo: 0, 1, 2 y 3.")
        elif not all(isinstance(v, str) and v.strip() for v in nombres.values()) or len({v.strip().lower() for v in nombres.values()}) != 4:
            r.mal("Cada grupo de `nombres` necesita un nombre de texto distinto.")
        else:
            r.ok("Cada grupo tiene su nombre.")
            if cs is not _FALTA and isinstance(cs, pd.DataFrame) and "cluster" in cs:
                if "segmento" not in cs or cs["segmento"].tolist() != [nombres[g] for g in cs["cluster"]]:
                    r.mal("Agrega a `clientes_seg` la columna `segmento` con el nombre de cada grupo (`map(nombres)`).")
                else:
                    r.ok("`clientes_seg` tiene la columna `segmento`.")
    r.fin()
    r = _Revision("Ejercicio 4 · Parte B")
    r.predicciones({
        "pred_kmeans_nombra": "e675bdd897ba87a607b7c344f97a5152cda452c1b1641515ea9436fd35397ada",
    })
    r.fin()


def _pca_ref():
    X = _X()
    Xc = X - X.mean(axis=0)
    val, vec = np.linalg.eigh(np.cov(Xc, rowvar=False))
    orden = np.argsort(val)[::-1]
    return X, Xc, val[orden], vec[:, orden]


def check_ejercicio_5():
    r = _Revision("Ejercicio 5 · Parte A")
    X, Xc, val, vec = _pca_ref()
    p = r.var("pca")
    if p is not _FALTA:
        if type(p).__name__ != "PCA" or not hasattr(p, "components_") or p.n_components_ != 2:
            r.mal("`pca` debería ser un `PCA(n_components=2)` ajustado con `X_esc`.")
        else:
            r.ok("`pca` es un PCA de 2 componentes.")
    _esc(r, "varianza_2", float(val[:2].sum() / val.sum()), "la suma de `explained_variance_ratio_`", tol=1e-6)
    comp = r.var("componentes")
    if comp is not _FALTA:
        comp = np.asarray(comp, dtype=float)
        ref = Xc @ vec[:, :2]
        if comp.shape != ref.shape:
            r.mal(f"`componentes` tiene forma {comp.shape} y se esperaba {ref.shape}.")
        elif not np.allclose(np.abs(comp), np.abs(ref), atol=1e-6):
            r.mal("Los valores de `componentes` no coinciden: usa `pca.transform(X_esc)`.")
        else:
            r.ok("`componentes` es correcto.")
    cargas = r.var("cargas")
    if cargas is not _FALTA:
        if not isinstance(cargas, pd.DataFrame) or [str(c) for c in cargas.columns] != ["PC1", "PC2"] or [str(i) for i in cargas.index] != VARIABLES:
            r.mal("`cargas` debería ser un DataFrame con `VARIABLES` como índice y las columnas PC1 y PC2.")
        elif not np.allclose(np.abs(cargas.to_numpy(float)), np.abs(vec[:, :2]), atol=1e-6):
            r.mal("Los valores de `cargas` no coinciden: son `pca.components_` traspuesto (`.T`).")
        else:
            r.ok("`cargas` es correcto.")
    ax = _grafico(r, "ax_pca")
    m = globals().get("km4")
    if ax is not None and hasattr(m, "cluster_centers_"):
        puntos = [c for c in ax.collections if len(c.get_offsets()) > 0]
        leyenda = ax.get_legend()
        if len(puntos) != 4 or sum(len(c.get_offsets()) for c in puntos) != len(X):
            r.mal("`ax_pca` debería tener un `scatter` por grupo (4 en total) con todos los clientes.")
        elif leyenda is None or len(leyenda.get_texts()) != 4:
            r.mal("Agrega una leyenda con los 4 grupos a `ax_pca`.")
        else:
            r.ok("`ax_pca` muestra los grupos en dos dimensiones.")
        _rotulos(r, "ax_pca", ax, None, "PC1", "PC2")
    r.fin()
    r = _Revision("Ejercicio 5 · Parte B")
    r.predicciones({
        "pred_pca_usa_etiquetas": "e675bdd897ba87a607b7c344f97a5152cda452c1b1641515ea9436fd35397ada",
    })
    r.fin()


def check_reto():
    r = _Revision("Reto final")
    m, e, nombres = globals().get("km4"), globals().get("escalador"), globals().get("nombres")
    if not hasattr(m, "cluster_centers_") or not hasattr(e, "mean_") or not isinstance(nombres, dict):
        r.mal("Primero resuelve los ejercicios 1 y 4 (`escalador`, `km4` y `nombres`).")
        r.fin()
        return
    A = clientes[VARIABLES].to_numpy(float)
    Xn = _esc_ref(A, clientes_nuevos[VARIABLES].to_numpy(float))
    dn = np.sqrt(_dist2(Xn, m.cluster_centers_))
    v = r.var("segmentos_nuevos", list)
    if v is not _FALTA:
        esperado = [nombres[int(g)] for g in dn.argmin(axis=1)]
        if len(v) != len(esperado):
            r.mal(f"`segmentos_nuevos` tiene {len(v)} elementos y se esperaban {len(esperado)}, uno por cliente nuevo.")
        elif v != esperado:
            r.mal("`segmentos_nuevos` no coincide: escala con `escalador.transform` (sin volver a ajustarlo), usa `km4.predict` y traduce con `nombres`.")
        else:
            r.ok("`segmentos_nuevos` es correcto.")
    _arr(r, "distancias_nuevos", dn.min(axis=1), "la distancia de cada cliente nuevo al centro más cercano (`km4.transform(...).min(axis=1)`)", tol=1e-6)
    d_train = np.sqrt(_dist2(_X(), m.cluster_centers_)).min(axis=1)
    umbral = float(np.percentile(d_train, 99))
    _esc(r, "umbral", umbral, "el percentil 99 de la distancia al centro más cercano de los clientes de `clientes`", tol=1e-6)
    _arr(r, "es_atipico", dn.min(axis=1) > umbral, "marca con True a quienes superan `umbral`", tipos="b")
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    m = globals().get("km4")
    if not hasattr(m, "cluster_centers_"):
        r.mal("Primero resuelve el ejercicio 1.")
        r.fin()
        return
    X = _X()
    et = _dist2(X, m.cluster_centers_).argmin(axis=1)
    s = _silueta(X, et)
    medias = [float(s[et == g].mean()) for g in range(4)]
    _ser(r, "silueta_por_cluster", medias, "la silueta media de cada grupo, con el número de grupo como índice", indice=[0, 1, 2, 3], tol=1e-6)
    _esc(r, "cluster_mas_debil", medias.index(min(medias)), "el grupo con menor silueta media", tol=0)
    r.fin()


print("✅ Setup listo. Datos generados, estilo aplicado y verificadores cargados.")

### 📦 Tus datos de hoy
`clientes`: 2000 clientes de un banco con su edad, saldo promedio (en soles), transacciones al mes, proporción de operaciones hechas desde la app (`uso_app`, de 0 a 1), gasto mensual con tarjeta (en soles) y proporción de pagos en efectivo (`pct_efectivo`). **No hay una columna objetivo**: nadie te dice a qué grupo pertenece cada cliente.

`clientes_nuevos`: 4 clientes que acaban de abrir su cuenta. `VARIABLES` tiene las seis columnas y `K_VALORES`, los números de grupos que vas a probar.

In [ ]:
print(clientes.head(), "\n")
print(clientes.describe().round(2))

---
## 1. K-means

### 📘 Concepto
En el aprendizaje **no supervisado** no hay una respuesta que predecir: se buscan estructuras en los datos, como grupos de clientes parecidos. **K-means** hace esto:
1. elige `k` centros iniciales;
2. asigna cada punto al centro más cercano;
3. mueve cada centro al promedio de sus puntos;
4. repite hasta que nada cambia.

Minimiza la **inercia**: la suma de las distancias al cuadrado de cada punto a su centro. Como el resultado depende del inicio, `n_init=10` lo repite 10 veces y se queda con la mejor, y `random_state` lo hace reproducible.

```python
from sklearn.cluster import KMeans
km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)
km.labels_            # grupo de cada fila: 0, 1, 2...
km.cluster_centers_   # coordenadas de los centros
km.inertia_           # inercia
```

Los números de grupo son solo etiquetas: el grupo 0 no es "mejor" ni "primero" que el 2. Como las distancias dependen de las unidades, primero se **escala** con `StandardScaler`. Aquí no hay entrenamiento y prueba: se ajusta con todos los clientes, pero se guarda el `escalador` para transformar clientes nuevos.

In [ ]:
from sklearn.cluster import KMeans

puntos_ej = np.array([[1, 1], [1.5, 2], [1, 1.5], [8, 8], [8.5, 9], [9, 8]])
km_ej = KMeans(n_clusters=2, n_init=10, random_state=0).fit(puntos_ej)
print(km_ej.labels_, km_ej.cluster_centers_.round(2), round(km_ej.inertia_, 2))

### ✍️ Tu turno · Ejercicio 1: cuatro grupos de clientes
**Parte A.**
1. `escalador`: un `StandardScaler` ajustado con `clientes[VARIABLES]`, y `X_esc`: esos datos transformados.
2. `km4`: un `KMeans(n_clusters=4, n_init=10, random_state=42)` ajustado con `X_esc`.
3. `etiquetas`: el grupo de cada cliente.
4. `tamanos`: una Series con cuántos clientes tiene cada grupo, con el número de grupo como índice ordenado.
5. `inercia_4`: la inercia de `km4`.

**Parte B.** Responde en `pred_numeros_cambian` con `"sí"` o `"no"`: si entrenas con otro `random_state` y salen los mismos grupos de clientes, ¿pueden cambiar los números que los identifican?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

Ajusta y transforma en dos pasos: `escalador = StandardScaler().fit(...)` y `X_esc = escalador.transform(...)`.
</details>

<details><summary>💡 Pista 2</summary>

`tamanos = pd.Series(etiquetas).value_counts().sort_index()`.
</details>

---
## 2. ¿Por qué escalar?

### 📘 Concepto
K-means mide distancias. Si una variable se mide en decenas de miles y otra entre 0 y 1, una diferencia de 1000 en la primera pesa muchísimo más que recorrer todo el rango de la segunda. Sin escalar, los grupos terminan cortados casi solo por la variable de mayor escala.

Para comparar dos agrupamientos se usa el **índice de Rand ajustado** (`adjusted_rand_score`): vale 1 si agrupan igual (aunque los números de grupo sean distintos) y cerca de 0 si coinciden como por azar.

In [ ]:
from sklearn.metrics import adjusted_rand_score

print(adjusted_rand_score([0, 0, 1, 1], [1, 1, 0, 0]))   # mismos grupos, otros números
print(adjusted_rand_score([0, 0, 1, 1], [0, 1, 0, 1]))   # grupos distintos

### ✍️ Tu turno · Ejercicio 2: agrupar sin escalar
**Parte A.**
1. `km_crudo`: el mismo K-means (4 grupos, `n_init=10`, `random_state=42`) ajustado con `clientes[VARIABLES]` **sin escalar**, y `etiquetas_crudo`: sus grupos.
2. `ari`: el índice de Rand ajustado entre `etiquetas` y `etiquetas_crudo`.
3. `desv_por_variable`: la desviación estándar de cada variable sin escalar, de mayor a menor.

Compara `clientes.groupby(etiquetas_crudo)[VARIABLES].mean()` con lo mismo para `etiquetas`. ¿Qué variable separa los grupos sin escalar?

**Parte B.** Responde en `pred_domina` con el nombre de la variable que domina el agrupamiento sin escalar.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

Es la misma línea del ejercicio 1, cambiando `X_esc` por `clientes[VARIABLES]`.
</details>

<details><summary>💡 Pista 2</summary>

`clientes[VARIABLES].std().sort_values(ascending=False)`.
</details>

---
## 3. ¿Cuántos grupos? El codo y la silueta

### 📘 Concepto
K-means necesita que le digas `k`. Dos herramientas ayudan a elegirlo:
- **Método del codo**: la inercia siempre baja al aumentar `k` (más centros, puntos más cerca). Se busca el "codo", el punto desde el que agregar grupos casi no mejora.
- **Silueta**: para cada punto compara la distancia media a los de su grupo (`a`) con la distancia media al grupo vecino más cercano (`b`): `(b − a) / máx(a, b)`. Va de −1 a 1; cerca de 1 es un punto bien ubicado. `silhouette_score` da el promedio, y se elige el `k` con la mayor silueta.

Ninguna regla es definitiva: el `k` final también depende de si los grupos se pueden usar en el negocio.

In [ ]:
from sklearn.metrics import silhouette_score

for k in [2, 3]:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=0).fit(puntos_ej)
    print(k, round(km_k.inertia_, 2), round(silhouette_score(puntos_ej, km_k.labels_), 3))

### ✍️ Tu turno · Ejercicio 3: elegir k
**Parte A.**
1. `evaluacion_k`: un DataFrame con `K_VALORES` como índice y las columnas `inercia` y `silueta` de un `KMeans(n_clusters=k, n_init=10, random_state=42)` ajustado con `X_esc` para cada `k`.
2. `k_silueta`: el `k` con mayor silueta.
3. `fig_k, (ax_codo, ax_silueta) = plt.subplots(1, 2, figsize=(10, 3.5))`: en `ax_codo` una línea con la inercia según `k` y en `ax_silueta` otra con la silueta, con marcadores, el eje x `k` y el eje y `Inercia` y `Silueta`.

¿Coinciden el codo y la silueta?

**Parte B.** Predice **sin ejecutar**: si `k` fuera igual a la cantidad de clientes, ¿cuánto valdría la inercia? Guárdalo en `pred_inercia_maxima_k` (un número decimal).

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

Recorre `K_VALORES` con un `for`, entrena en cada vuelta y guarda `[km.inertia_, silhouette_score(X_esc, km.labels_)]` en una lista.
</details>

<details><summary>💡 Pista 2</summary>

`evaluacion_k = pd.DataFrame(filas, index=K_VALORES, columns=["inercia", "silueta"])` y `k_silueta = evaluacion_k["silueta"].idxmax()`.
</details>

---
## 4. Interpretar los segmentos

### 📘 Concepto
K-means entrega números, no significados. Para interpretar un grupo se mira su **perfil**:
- el promedio de cada variable **original** por grupo, en sus unidades (soles, años), para contarlo;
- el promedio de cada variable **escalada** por grupo: como está en desviaciones estándar respecto del total, muestra qué distingue a cada grupo (+1 es bastante más que el promedio, −1 bastante menos).

Con eso se le pone un nombre que describa y sirva para actuar (por ejemplo, "digitales de bajo saldo"). Revisa también el tamaño: un segmento de 20 clientes rara vez justifica una estrategia propia.

In [ ]:
tabla_ej = pd.DataFrame({"gasto": [10, 12, 90, 95, 11], "visitas": [1, 2, 8, 9, 1], "grupo": [0, 0, 1, 1, 0]})
print(tabla_ej.groupby("grupo").agg(gasto=("gasto", "mean"), visitas=("visitas", "mean"), clientes=("gasto", "size")))

### ✍️ Tu turno · Ejercicio 4: ponle nombre a cada grupo
**Parte A.**
1. `clientes_seg`: una copia de `clientes` con la columna `cluster` (las `etiquetas` de `km4`).
2. `perfil`: un DataFrame con un grupo por fila (índice 0 a 3) y el promedio de cada variable de `VARIABLES`, más la columna `clientes` con la cantidad de clientes del grupo.
3. `perfil_z`: lo mismo con `X_esc` (conviértelo en un DataFrame con las columnas de `VARIABLES`), sin la columna `clientes`.
4. `nombres`: un diccionario `{número de grupo: nombre}` con un nombre corto y distinto para cada grupo, según su perfil. Agrega a `clientes_seg` la columna `segmento` con ese nombre.

**Parte B.** Responde en `pred_kmeans_nombra` con `"sí"` o `"no"`: ¿K-means te dice qué significa cada grupo?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

`clientes_seg.groupby("cluster")[VARIABLES].mean()` da los promedios. La cantidad sale de `.size()` o de `value_counts()`.
</details>

<details><summary>💡 Pista 2</summary>

Para `perfil_z`: `pd.DataFrame(X_esc, columns=VARIABLES).groupby(etiquetas).mean()`. Para los nombres, mira en `perfil_z` los valores más altos y más bajos de cada fila.
</details>

---
## 5. PCA: ver seis variables en dos

### 📘 Concepto
No se pueden dibujar seis dimensiones. El **análisis de componentes principales** (PCA) busca nuevas variables, combinaciones de las originales, que capturan la mayor variación posible:
- `PC1` es la dirección en la que los datos más varían; `PC2`, la siguiente, perpendicular a la primera;
- `explained_variance_ratio_` dice qué proporción de la variación total captura cada componente;
- `components_` tiene las **cargas**: cuánto aporta cada variable original a cada componente.

PCA también necesita datos escalados y **no usa** los grupos: es otra técnica no supervisada. Aquí sirve para dibujar los grupos de K-means en un plano.

In [ ]:
from sklearn.decomposition import PCA

nube_ej = np.random.default_rng(0).normal(size=(200, 1)) @ np.array([[1.0, 0.9, 0.1]]) + np.random.default_rng(1).normal(scale=0.2, size=(200, 3))
pca_ej = PCA(n_components=2).fit(nube_ej)
print(pca_ej.explained_variance_ratio_.round(3))
print(pca_ej.components_.round(2))

### ✍️ Tu turno · Ejercicio 5: el mapa de los segmentos
**Parte A.**
1. `pca`: un `PCA(n_components=2)` ajustado con `X_esc`, y `componentes`: `X_esc` transformado.
2. `varianza_2`: la proporción de la variación total que capturan los dos componentes juntos.
3. `cargas`: un DataFrame con las cargas, `VARIABLES` como índice y las columnas `PC1` y `PC2`.
4. `fig_pca, ax_pca`: un `scatter` por grupo (con `label` para la leyenda, por ejemplo el nombre de `nombres`), con PC1 en el eje x y PC2 en el eje y, puntos pequeños y algo transparentes (`s=8, alpha=0.5`), leyenda y ejes rotulados `PC1` y `PC2`.

Mira `cargas`: ¿qué variables pesan más en PC1? ¿Los grupos se ven separados?

**Parte B.** Responde en `pred_pca_usa_etiquetas` con `"sí"` o `"no"`: ¿PCA usa los grupos de K-means para encontrar los componentes?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_5()

<details><summary>💡 Pista 1</summary>

`pca.explained_variance_ratio_.sum()` da la proporción. Para el gráfico, recorre los grupos con un `for` y filtra `componentes[etiquetas == g]`.
</details>

<details><summary>💡 Pista 2</summary>

`cargas = pd.DataFrame(pca.components_.T, index=VARIABLES, columns=["PC1", "PC2"])`.
</details>

---
## 🏋️ Reto final: asignar clientes nuevos
Un segmento solo sirve si puedes ubicar a los clientes que llegan después.
1. `segmentos_nuevos`: una lista con el nombre de segmento de cada cliente de `clientes_nuevos`. Escala con el `escalador` ya ajustado (`transform`, **sin** volver a ajustarlo), usa `km4.predict` y traduce con `nombres`.
2. `distancias_nuevos`: la distancia de cada cliente nuevo al centro más cercano. `km4.transform(...)` da la distancia a cada centro.
3. `umbral`: el percentil 99 de esa misma distancia para los clientes de `clientes` (usa `X_esc`).
4. `es_atipico`: un array booleano que marca a los clientes nuevos cuya distancia supera `umbral`.

K-means siempre asigna un grupo, aunque el cliente no se parezca a ninguno. ¿Qué harías con un cliente atípico?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

`km4.transform(X)` devuelve una matriz con una columna por centro; `.min(axis=1)` se queda con la distancia al más cercano.
</details>

<details><summary>💡 Pista 2</summary>

`np.percentile(km4.transform(X_esc).min(axis=1), 99)` da el umbral.
</details>

---
## 🚀 Nivel pro (opcional): la silueta de cada grupo
La silueta promedio puede esconder un grupo mal formado. `silhouette_samples(X_esc, etiquetas)` da la silueta de **cada** cliente. Crea `silueta_por_cluster`: una Series con la silueta media de cada grupo (índice 0 a 3), y `cluster_mas_debil`: el grupo con la menor. ¿Qué segmento es ese según `nombres`? ¿Tiene clientes con silueta negativa?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## 🧱 Avance del proyecto: P4 · segmentación

**Qué hacer**
1. Elige **qué** segmentar según tus datos: entidades financieras, regiones o productos, descritos por su perfil de reclamos (por ejemplo, reclamos por cada 10 000 clientes, reparto por motivo y porcentaje resuelto a favor del consumidor). Usa las tablas limpias de P2.
2. En `notebooks/05_segmentacion.ipynb` (o una sección al final de `03_eda_visualizacion.ipynb`), arma una tabla con una fila por unidad y solo variables comparables: tasas y proporciones, no totales que dependan del tamaño.
3. Escala, prueba varios `k` con el codo y la silueta, y elige uno que puedas explicar. Si tienes pocas filas (por ejemplo, 25 regiones), prefiere pocos grupos y revisa que ninguno quede con 1 o 2 unidades.
4. Interpreta cada grupo con su perfil, ponle un nombre y dibújalo con PCA o, si alcanza, con las dos variables que más lo distinguen.
5. Anota en el README qué segmentos encontraste, qué los distingue y cómo cambiaría tu recomendación de P4 según el segmento.

**Por qué lo haría un analista**
Una recomendación única para todos suele ser mediocre para cada uno. Segmentar permite priorizar: atacar primero el grupo donde el problema es mayor o la acción es más barata.

**Cómo debe verse el resultado**
Una tabla de perfiles con nombres que se entienden sin ver el código, un gráfico de los segmentos con título-conclusión y dos o tres líneas en el README. Estas tablas irán a tu base SQL (S25) y a tu dashboard (S27).

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Explicar qué hace K-means y qué es la inercia.
- [ ] Explicar por qué hay que escalar antes de agrupar.
- [ ] Elegir `k` con el codo y la silueta, y decir por qué ninguno es definitivo.
- [ ] Interpretar un segmento con su perfil original y escalado, y nombrarlo.
- [ ] Explicar qué muestra PCA y qué son las cargas.
- [ ] Asignar clientes nuevos a un segmento y detectar los que no encajan.

**Próxima sesión (S25):** SQL desde Python: `sqlite3`, `pd.read_sql`, `JOIN`, `GROUP BY`, CTEs y funciones de ventana.